<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">STEP-BY-STEP GUIDE TO RUN LLM MODELS ON A DISTRIBUTED CLUSTER (HEAD)</h2>

<h3 style="color:#44D62C; text-align:left;">Project Overview</h3>

Razer AIKit is Razer's AI developer environment built to simplify and accelerate machine learning workflows on high-performance Razer hardware. It offers a plug-and-play experience for running Large Language Models (LLMs) and other AI models locally using an optimized stack.

<h3 style="color:#44D62C; text-align:left;">Objectives of This Guide</h3>

This guide focuses on distributed model inferencing with Razer AIKit.  
It enables you to run models that are too large to fit into a single machine’s GPU memory (e.g., Qwen3-14B, LLaMA 70B) using `rzr-aikit`'s built-in cluster support.


<h3 style="color:#44D62C; text-align:left;">🔍 1. Get Model Information</h3>

We begin by inspecting the configuration of the model we intend to serve.

In this guide, we’ll use <code>Qwen/Qwen3-14B</code> — a powerful large language model that typically **exceeds the memory limits of a single consumer GPU**.

The command below will display key metadata such as model size, base precision, and compatibility.  
Pay special attention to the <strong>Compatibility</strong> section, which will indicate whether distributed deployment is required.

In [ ]:
rzr-aikit model info Qwen/Qwen3-14B

<h3 style="color:#44D62C; text-align:left;">📥 2. Download the Model</h3>

Before running a model in distributed mode, ensure it is downloaded on every participating machine.

This command will poll the model from Hugging Face into the local Razer environment cache.

In [ ]:
rzr-aikit model download Qwen/Qwen3-14B

<h3 style="color:#44D62C; text-align:left;">🔗 3. Start Cluster on Head Node</h3>

Next, start the cluster on your <strong>head node</strong>.

This node will coordinate scheduling, memory partitioning, and request routing across the cluster.

Use the <code>--ifname</code> flag to specify the network interface used for inter-node communication (e.g. <code>enp47s0</code>). You can identify it with <code>ip</code> or <code>ifconfig</code>.


In [ ]:
ip -f inet -4 -o addr

In [ ]:

rzr-aikit cluster run --ifname enp47s0 --metrics-export-port 8080

<h3 style="color:#44D62C; text-align:left;">🛰️ 4. Check Cluster Status</h3>

For distributed model deployment, at least <strong>two active nodes</strong> are required — one head and at least one worker node.

After starting the cluster on the head node, use the following command to verify cluster status.  
It will list all connected nodes and resource availability.

You can also view a real-time dashboard in your browser at: http://localhost:8265


In [ ]:
rzr-aikit cluster status

If no worker nodes are shown, follow Guide <strong>2b_(Node)_Distributed_Inferencing<strong>

<h3 style="color:#44D62C; text-align:left;">🚀 5. Run the Model</h3>

Once your cluster is active with all required nodes, you're ready to launch the model in distributed mode.

The following command starts the model with pipeline parallelism across the cluster nodes.

<div style="border: 2px solid #44D62C; border-radius: 8px; padding: 15px; margin: 10px 0; box-shadow: 0 2px 4px rgba(68, 214, 44, 0.2);">
<h4 style="color: #44D62C; margin-top: 0;">💡 GPU Configuration Helper (Optional)</h4>
<p><strong>Note:</strong> Configuration and distribution of load across multiple GPUs and multiple Nodes can be challenging. There are 3 helper scripts that will assist you in this situation:</p>
<ul>
<li><code style="color: #44D62C; padding: 2px 4px; border-radius: 3px;">gpu-discover</code> - Discover available GPUs in your node/cluster</li>
<li><code style="color: #44D62C; padding: 2px 4px; border-radius: 3px;">gpu-select [model-name]</code> - Select optimal GPU configuration for your model</li>
<li><code style="color: #44D62C; padding: 2px 4px; border-radius: 3px;">gpu-filter</code> - Prepare specific GPUs for AI workloads</li>
</ul>
</div>

In [ ]:
gpu-discover --distributed

In [ ]:
gpu-select Qwen/Qwen3-14B

# Use '--restrict-gpus 0,1,2,3' to limit gpu-select to choosing only from this restricted set of GPUs

In [ ]:
#Replace this command with the recommendation provided by gpu-select above.

# Use VLLM_GPU_ORDER to reorder GPUs and assign the most powerful GPUs
# to the most compute-intensive parts of the model.
#
# Example:
# export VLLM_GPU_ORDER="GPU-186cbfe6-211d-fe1b-8255-a68e5c827a33,GPU-53e0cff4-42a4-601a-e08b-e2543b83bf74"

rzr-aikit model run Qwen/Qwen3-14B --enforce-eager --distributed-executor-backend ray

<h3 style="color:#44D62C; text-align:left;">💬 6. Generate Text</h3>

Once the model is running, you can send prompts and receive completions using the `generate` command.

This is ideal for interactive testing and quick validation of the model’s behavior.

In [ ]:
rzr-aikit model generate "Explain quantum computing to a 12-year-old."

<h3 style="color:#44D62C; text-align:left;">📊 7. Benchmark the Model</h3>

Use the benchmarking tool to evaluate how the model performs on your specific hardware.

This is useful for measuring:

- First token latency (TTFT)
- Throughput per output token (TPOT)
- Inference time per prompt (ITL)
- End-to-end latency (E2EL)

In [ ]:
vllm bench serve \
  --model Qwen/Qwen3-14B \
  --dataset-name sharegpt \
  --dataset-path ~/benchmarks/ShareGPT_300_conversations.json \
  --request-rate 10.0 \
  --host 127.0.0.1 \
  --port 8000 \
  --num-prompts 10 \
  --percentile-metrics ttft,tpot,itl,e2el

<h3 style="color:#44D62C; text-align:left;">🛑 8. Stop the Model</h3>

Use this command to gracefully shut down a running model service.  
This frees GPU and memory resources, and ensures clean shutdown of any background inference processes.

In [ ]:
rzr-aikit model stop

<h3 style="color:#44D62C; text-align:left;">🛑 9. Stop the Cluster</h3>

Once you’re finished with distributed inferencing, you can gracefully shut down the cluster on both the head and worker nodes to free up GPU and system resources.

In [ ]:
rzr-aikit cluster stop

---

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>

By following this guide, you've learned how to set up and run large-scale LLMs across multiple machines using the `rzr-aikit` and Razer AIKit's distributed cluster environment.

##### What You've Achieved

- Retrieved model metadata and verified the need for distributed deployment
- Downloaded a compatible model on all cluster nodes
- Started a distributed model-serving cluster from the head node
- Verified cluster health and accessed real-time dashboard insights
- Launched <code>Qwen/Qwen3-14B</code> across nodes with pipeline parallelism
- Benchmarked model performance on your cluster using token-level metrics
- Stopped the distributed model cleanly and released all resources

##### Use Cases This Enables

- Serving very large models beyond single-GPU capacity
- Running inference workloads across multiple machines
- Scaling LLM serving performance for latency- or throughput-sensitive applications
- Prototyping distributed architectures for deployment in edge, lab, or on-prem environments

This forms a solid foundation for future exploration of advanced features like multi-model orchestration, cluster autoscaling, and fine-tuning at scale.

